In [ ]:
import psycopg2
import psycopg2.extras

def get_near_bus_routes(lon, lat, radius=1600,
                        dbname="osm", user="postgres",
                        password="67500", host="localhost", port=5433):
    """
    Find nearby bus routes.
    Boarding = closest point on line (gtfs_shapes).
    """
    query = """
    WITH my_point AS (
      SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography AS geom
    )
    SELECT s.shape_id,
           s.route_type,
           'N/A'::text AS stop_name,
           ST_AsGeoJSON(ST_ClosestPoint(s.geom, p.geom::geometry)) AS boarding_point,
           ST_Distance(s.geom::geography, p.geom) AS dist_meters
    FROM gtfs_shapes s, my_point p
    WHERE s.route_type = 3   -- bus
      AND ST_DWithin(s.geom::geography, p.geom, %s)
    ORDER BY dist_meters;
    """

    conn = psycopg2.connect(dbname=dbname, user=user,
                            password=password, host=host, port=port)
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cur.execute(query, (lon, lat, radius))
    rows = cur.fetchall()
    cur.close()
    conn.close()

    return [
        {
            "shape_id": row["shape_id"],
            "route_type": row["route_type"],
            "boarding_name": row["stop_name"],  # "N/A" for bus
            "boarding_geom": row["boarding_point"],  # GeoJSON
            "dist_meters": float(row["dist_meters"])
        }
        for row in rows
    ]


In [ ]:
def get_near_tram_routes(lon, lat, radius=1600,
                         dbname="osm", user="postgres",
                         password="67500", host="localhost", port=5433):
    """
    Find nearby tram/rail routes.
    Boarding = nearest stop per route.
    """
    query = """
    WITH my_point AS (
      SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography AS geom
    ),
    ranked AS (
      SELECT r.route_id,
             r.route_type,
             s.stop_id,
             s.stop_name,
             s.geom AS boarding_point,
             ST_Distance(s.geom::geography, p.geom) AS dist_meters,
             ROW_NUMBER() OVER (PARTITION BY r.route_id ORDER BY s.geom <-> p.geom) AS rn
      FROM stops s
      JOIN stop_routes sr ON s.stop_id = sr.stop_id
      JOIN routes r ON sr.route_id = r.route_id
      , my_point p
      WHERE r.route_type IN (0,1,2)  -- tram, subway, rail
        AND ST_DWithin(s.geom::geography, p.geom, %s)
    )
    SELECT route_id, route_type, stop_id, stop_name,
           ST_AsGeoJSON(boarding_point) AS boarding_point,
           dist_meters
    FROM ranked
    WHERE rn = 1
    ORDER BY dist_meters;
    """

    conn = psycopg2.connect(dbname=dbname, user=user,
                            password=password, host=host, port=port)
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cur.execute(query, (lon, lat, radius))
    rows = cur.fetchall()
    cur.close()
    conn.close()

    return [
        {
            "route_id": row["route_id"],
            "route_type": row["route_type"],
            "stop_id": row["stop_id"],
            "boarding_name": row["stop_name"],
            "boarding_geom": row["boarding_point"],  # GeoJSON
            "dist_meters": float(row["dist_meters"])
        }
        for row in rows
    ]


In [ ]:
def get_route_path(lon, lat, boarding_geom,
                   dbname="osm", user="postgres",
                   password="67500", host="localhost", port=5433):
    """
    Compute walking path from (lon, lat) to a boarding point (GeoJSON point).
    """
    query = """
    WITH input AS (
      SELECT ST_SetSRID(ST_Point(%s, %s), 4326) AS geom
    ),
    start_node AS (
      SELECT id
      FROM ways_vertices_pgr
      ORDER BY the_geom <-> (SELECT geom FROM input)
      LIMIT 1
    ),
    target_node AS (
      SELECT id
      FROM ways_vertices_pgr
      ORDER BY the_geom <-> ST_SetSRID(ST_GeomFromGeoJSON(%s), 4326)
      LIMIT 1
    ),
    walk_path AS (
      SELECT e.the_geom, e.length_m
      FROM pgr_dijkstra(
        'SELECT gid AS id, source, target, length_m AS cost FROM ways',
        (SELECT id FROM start_node),
        (SELECT id FROM target_node),
        false
      ) dj
      JOIN ways e ON dj.edge = e.gid
    )
    SELECT ST_AsGeoJSON(ST_LineMerge(ST_Union(w.the_geom))) AS walk_geom,
           SUM(w.length_m) AS walk_distance_m
    FROM walk_path w;
    """

    conn = psycopg2.connect(dbname=dbname, user=user,
                            password=password, host=host, port=port)
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cur.execute(query, (lon, lat, boarding_geom))
    row = cur.fetchone()
    cur.close()
    conn.close()

    if row:
        return {
            "walk_geom": row["walk_geom"],
            "walk_distance_m": float(row["walk_distance_m"]) if row["walk_distance_m"] else None
        }
    else:
        return None


In [ ]:
lon, lat = 29.96139328537071, 31.22968895248673

buses = get_near_bus_routes(lon, lat)
trams = get_near_tram_routes(lon, lat)

# merge results
candidates = buses + trams
candidates = sorted(candidates, key=lambda x: x["dist_meters"])

for c in candidates:
    print(c)


In [8]:
best

{'shape_id': 'abuqir',
 'route_type': 3,
 'boarding_name': 'N/A',
 'boarding_geom': '{"type":"Point","coordinates":[29.957655872,31.233196371]}',
 'dist_meters': 521.29826406}